# Bài 2 (trang PDF 15–17, Titanic) – Câu 6

**Sinh viên:** Trần Chí Nguyên  
**MSSV:** 6451071052

Notebook chạy độc lập trên Google Colab. Dữ liệu công khai được tải tự động khi chạy cell đầu tiên.

In [ ]:
import requests, numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

print("Trần Chí Nguyên - 6451071052")
FILE_ID = "1TCnJqaowkv0caJTJrmjoqb0AB2bMvgxP"
def load_titanic():
    path = Path("/content/titanic.csv")
    if not path.exists():
        response = requests.get("https://drive.google.com/uc", params={"export": "download", "id": FILE_ID}, timeout=90)
        response.raise_for_status()
        path.write_bytes(response.content)
    data = pd.read_csv(path)
    data.columns = [str(c).strip().lower().replace("_", ".") for c in data.columns]
    return data
df = load_titanic()
print("Kích thước dữ liệu:", df.shape)
display(df.head())

## Thực hiện yêu cầu

Cell dưới đây chứa lời giải cho câu này.

In [ ]:
work = df.copy()
# Giữ ký tự boong từ cabin trước khi bỏ các cột ít giá trị phân tích.
work["cabin_deck"] = work["cabin"].astype("string").str[0]
work = work.drop(columns=[c for c in ["body", "cabin", "boat", "home.dest", "ticket"] if c in work.columns])
if "age" in work:
    work["age"] = pd.to_numeric(work["age"], errors="coerce")
    work["age"] = work["age"].fillna(work["age"].median())
if "embarked" in work:
    work["embarked"] = work["embarked"].fillna(work["embarked"].mode().iat[0])
if "fare" in work:
    work["fare"] = pd.to_numeric(work["fare"], errors="coerce").fillna(work["fare"].median()).astype("int64")
work["title"] = work["name"].str.extract(r",\s*([^.]*)\.", expand=False).str.strip()
work["sex_code"] = work["sex"].str.lower().map({"male": 0, "female": 1})
work["embarked_code"] = work["embarked"].map({"S": 0, "C": 1, "Q": 2})
work["age"] = work["age"].round().astype("int64")
age_bins = [-np.inf, 10, 18, 22, 27, 33, 40, 66, np.inf]
work["age_group"] = pd.cut(work["age"], bins=age_bins, labels=False, include_lowest=True) + 1
fare_bins = [-np.inf, 7.91, 14.454, 31, 99, 250, np.inf]
work["fare_group"] = pd.cut(work["fare"], bins=fare_bins, labels=False, include_lowest=True) + 1
work["age_class"] = work["age"] * work["pclass"]
work["fare_per_person"] = work["fare"] / (work["sibsp"].fillna(0) + work["parch"].fillna(0) + 1)
display(work.head())
print("Kích thước sau tiền xử lý:", work.shape)
print("Giá trị rỗng còn lại theo cột:"); display(work.isna().sum().sort_values(ascending=False).head(10))